# 00 · Smoke test

Checks that every model in the project runs end to end on a GPU before any real experiment starts.

- Downloads ULB and checks it is the file we expect (284,807 rows, 492 frauds).
- Runs TabPFN v2, TabICLv2 and XGBoost once on a small sample.
- Times TabPFN v2 and TabICLv2 at the largest context we plan to use (10K rows), so we can estimate GPU-hours for the full sweep.

**The PR-AUC numbers here are not results.** The sample is small and the fraud rate is inflated. They only show that the pipeline produces sensible numbers.

Run on Colab with **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Is there a GPU? TabPFN and TabICL are transformers and are far slower on CPU.
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# 2. Get the code: clone the repo the first time, pull updates after that.
#    Then install the pinned libraries and make src/ importable.
import os, sys

REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, f"{REPO_DIR}/src")
print("Code ready at", REPO_DIR)

If pip prints a message asking you to restart the session, do **Runtime → Restart session** and run from cell 1 again. That happens when pip upgrades a library Colab had already loaded.

In [ ]:
# 3. Download ULB (public Kaggle dataset; kagglehub needs no token for it) and check it.
import kagglehub, pandas as pd

ulb_dir = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(f"{ulb_dir}/creditcard.csv")

n_rows, n_cols = df.shape
n_fraud = int(df["Class"].sum())
print(f"rows={n_rows:,}  columns={n_cols}  frauds={n_fraud}  fraud rate={n_fraud / n_rows:.4%}")
print("missing values:", int(df.isna().sum().sum()))

# If these fail, we downloaded a different file from the one in the proposal.
assert n_rows == 284_807 and n_fraud == 492, "Unexpected dataset size"
assert list(df.columns[:2]) == ["Time", "V1"] and df.columns[-1] == "Class"

In [ ]:
# 4. A small sample just for the smoke test: all 492 frauds + 9,508 legitimate rows
#    (10,000 rows, fraud rate about 4.9%), split 50/50 with the fraud rate kept equal
#    in both halves. The real splits and prevalence levels come in task T2.
import numpy as np
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
fraud_idx = np.flatnonzero(df["Class"].values == 1)
legit_idx = rng.choice(np.flatnonzero(df["Class"].values == 0), size=9_508, replace=False)
sample = df.iloc[np.concatenate([fraud_idx, legit_idx])]

X = sample.drop(columns="Class").to_numpy(dtype=np.float32)
y = sample["Class"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)
print(f"train: {len(y_tr):,} rows, {y_tr.sum()} frauds | test: {len(y_te):,} rows, {y_te.sum()} frauds")

In [ ]:
# 5. One helper that fits a model, scores it and records time and GPU memory,
#    so all three models are measured the same way.
import time
from sklearn.metrics import average_precision_score, roc_auc_score

def run(name, model, X_tr, y_tr, X_te, y_te):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)                  # for TabPFN / TabICL, "fit" only stores the context rows
    t1 = time.perf_counter()
    p = model.predict_proba(X_te)[:, 1]    # probability of fraud for each test row
    t2 = time.perf_counter()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else float("nan")
    return dict(
        model=name,
        pr_auc=round(average_precision_score(y_te, p), 4),  # headline metric
        roc_auc=round(roc_auc_score(y_te, p), 4),
        fit_s=round(t1 - t0, 2),
        predict_s=round(t2 - t1, 2),
        peak_gpu_gb=round(peak_gb, 2),
    )

In [ ]:
# 6. Run all three models once. The first TabPFN / TabICL call downloads the weights
#    (a few hundred MB), so it is slower than later calls.
from fraudicl.models import make_tabpfn_v2, make_tabicl_v2, make_xgboost

rows = [
    run("TabPFN v2", make_tabpfn_v2(seed=0), X_tr, y_tr, X_te, y_te),
    run("TabICLv2", make_tabicl_v2(seed=0), X_tr, y_tr, X_te, y_te),
    run("XGBoost (default params)", make_xgboost(seed=0), X_tr, y_tr, X_te, y_te),
]
smoke = pd.DataFrame(rows)
smoke

**What to look for:** all three rows filled in, PR-AUC well above the 0.049 a random guess would score at this fraud rate, and no errors. Fit time for the two TFMs should be close to zero, because they don't train.

In [ ]:
# 7. Timing probe at full scale: a 10,000-row context (the most TabPFN v2 accepts)
#    and 20,000 rows to predict, taken from the full dataset. This tells us how long
#    the real sweep will take. Scores are not reported here, only time and memory.
idx = rng.choice(len(df), size=30_000, replace=False)
X_big = df.drop(columns="Class").to_numpy(dtype=np.float32)[idx]
y_big = df["Class"].to_numpy()[idx]
# make sure the context has some frauds, otherwise the models see one class only
ctx = np.concatenate([np.flatnonzero(y_big == 1)[:30], np.flatnonzero(y_big == 0)[:9_970]])
rest = np.setdiff1d(np.arange(len(idx)), ctx)[:20_000]

probe = []
for name, make in [("TabPFN v2", make_tabpfn_v2), ("TabICLv2", make_tabicl_v2)]:
    r = run(name, make(seed=0), X_big[ctx], y_big[ctx], X_big[rest], y_big[rest])
    r["seconds_per_1k_predictions"] = round(1000 * r["predict_s"] / len(rest), 3)
    probe.append({k: r[k] for k in ["model", "fit_s", "predict_s", "seconds_per_1k_predictions", "peak_gpu_gb"]})
probe = pd.DataFrame(probe)
probe

**What this tells us:** seconds per 1,000 predictions at the largest context. The full ULB test split has about 57,000 rows, so one TFM run at k = 10K takes roughly 57 × that number. We use it in task T5 to size the sweep.

In [ ]:
# 8. Save a small summary and print versions, so the result can be pasted back
#    into the plan and committed.
import json, platform, importlib.metadata as md_

versions = {p: md_.version(p) for p in ["tabpfn", "tabicl", "xgboost", "scikit-learn", "torch", "numpy", "pandas"]}
summary = dict(
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    python=platform.python_version(),
    versions=versions,
    smoke=smoke.to_dict(orient="records"),
    timing_probe=probe.to_dict(orient="records"),
)
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
with open(f"{REPO_DIR}/results/smoke_test.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))